<a href="https://colab.research.google.com/github/dystaSatria/Deep-Learning/blob/main/Internship%20Projects/recall/recall.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import DenseNet121, DenseNet169, DenseNet201
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve
import seaborn as sns
from typing import Tuple, List, Dict, Optional

class CustomDenseNet:
    """
    Custom DenseNet implementasyonu recall optimizasyonu ile
    """

    def __init__(self,
                 input_shape: Tuple[int, int, int] = (224, 224, 3),
                 num_classes: int = 2,
                 growth_rate: int = 32,
                 depth: List[int] = [6, 12, 24, 16],
                 compression: float = 0.5):
        """
        DenseNet parametreleri

        Args:
            input_shape: Giriş boyutu (height, width, channels)
            num_classes: Sınıf sayısı
            growth_rate: Her katmanda eklenen feature map sayısı
            depth: Her dense block'taki katman sayısı listesi
            compression: Transition layer'da sıkıştırma oranı
        """
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.growth_rate = growth_rate
        self.depth = depth
        self.compression = compression
        self.model = None

    def dense_layer(self, x, growth_rate: int) -> tf.Tensor:
        """Tek bir dense layer (BN-ReLU-Conv1x1-BN-ReLU-Conv3x3)"""
        # Bottleneck layer (1x1 conv)
        x1 = layers.BatchNormalization()(x)
        x1 = layers.Activation('relu')(x1)
        x1 = layers.Conv2D(4 * growth_rate, 1, use_bias=False)(x1)

        # 3x3 convolution
        x1 = layers.BatchNormalization()(x1)
        x1 = layers.Activation('relu')(x1)
        x1 = layers.Conv2D(growth_rate, 3, padding='same', use_bias=False)(x1)

        # Dense connection: concatenate input dengan output
        return layers.concatenate([x, x1])

    def dense_block(self, x, num_layers: int, growth_rate: int) -> tf.Tensor:
        """Dense block - berisi multiple dense layers dengan dense connectivity"""
        for i in range(num_layers):
            x = self.dense_layer(x, growth_rate)
        return x

    def transition_layer(self, x, compression: float) -> tf.Tensor:
        """Transition layer - mengurangi dimensi antar dense blocks"""
        num_filters = int(x.shape[-1] * compression)

        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Conv2D(num_filters, 1, use_bias=False)(x)
        x = layers.AveragePooling2D(2, strides=2)(x)

        return x

    def build_model(self) -> keras.Model:
        """DenseNet modelini oluştur"""
        inputs = keras.Input(shape=self.input_shape)

        # İlk convolution layer
        x = layers.Conv2D(64, 7, strides=2, padding='same', use_bias=False)(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.MaxPooling2D(3, strides=2, padding='same')(x)

        # Dense blocks ve transition layers
        for i, num_layers in enumerate(self.depth):
            x = self.dense_block(x, num_layers, self.growth_rate)

            # Son block haricinde transition layer ekle
            if i < len(self.depth) - 1:
                x = self.transition_layer(x, self.compression)

        # Final layers
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.GlobalAveragePooling2D()(x)

        # Classification head
        outputs = layers.Dense(self.num_classes, activation='softmax')(x)

        self.model = keras.Model(inputs, outputs, name='custom_densenet')
        return self.model

class RecallOptimizer:
    """
    Recall optimizasyonu için yardımcı sınıf
    """

    @staticmethod
    def focal_loss(alpha: float = 0.25, gamma: float = 2.0):
        """
        Focal Loss - imbalanced dataset'ler için etkili
        Hard-to-classify örneklere daha fazla odaklanır
        """
        def focal_loss_fn(y_true, y_pred):
            epsilon = tf.keras.backend.epsilon()
            y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)

            # Cross entropy hesapla
            ce = -y_true * tf.math.log(y_pred)

            # Focal weight hesapla
            weight = alpha * y_true * tf.pow((1 - y_pred), gamma)

            # Focal loss
            fl = weight * ce
            return tf.reduce_mean(tf.reduce_sum(fl, axis=1))

        return focal_loss_fn

    @staticmethod
    def weighted_binary_crossentropy(pos_weight: float = 1.0):
        """
        Weighted Binary Cross-entropy - pozitif sınıfa daha fazla ağırlık verir
        """
        def weighted_loss(y_true, y_pred):
            epsilon = tf.keras.backend.epsilon()
            y_pred = tf.clip_by_value(y_pred, epsilon, 1 - epsilon)

            # Weighted cross-entropy
            loss = -(pos_weight * y_true * tf.math.log(y_pred) +
                    (1 - y_true) * tf.math.log(1 - y_pred))

            return tf.reduce_mean(loss)

        return weighted_loss

    @staticmethod
    def recall_metric(threshold: float = 0.5):
        """Custom recall metric with adjustable threshold"""
        def recall_fn(y_true, y_pred):
            y_pred_binary = tf.cast(y_pred > threshold, tf.float32)

            true_positives = tf.reduce_sum(y_true * y_pred_binary)
            possible_positives = tf.reduce_sum(y_true)

            recall = true_positives / (possible_positives + tf.keras.backend.epsilon())
            return recall

        return recall_fn

class DenseNetTrainer:
    """
    DenseNet eğitimi ve recall optimizasyonu için ana sınıf
    """

    def __init__(self,
                 input_shape: Tuple[int, int, int] = (224, 224, 3),
                 num_classes: int = 2,
                 use_pretrained: bool = True):
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.use_pretrained = use_pretrained
        self.model = None
        self.history = None

    def create_model(self,
                     model_type: str = 'densenet121',
                     custom_top: bool = True) -> keras.Model:
        """
        DenseNet model oluştur (pretrained veya custom)
        """
        if self.use_pretrained:
            # Pretrained DenseNet kullan
            if model_type == 'densenet121':
                base_model = DenseNet121(weights='imagenet',
                                       include_top=False,
                                       input_shape=self.input_shape)
            elif model_type == 'densenet169':
                base_model = DenseNet169(weights='imagenet',
                                       include_top=False,
                                       input_shape=self.input_shape)
            else:
                base_model = DenseNet201(weights='imagenet',
                                       include_top=False,
                                       input_shape=self.input_shape)

            if custom_top:
                # Custom classification head ekle
                inputs = keras.Input(shape=self.input_shape)
                x = base_model(inputs, training=False)
                x = layers.GlobalAveragePooling2D()(x)
                x = layers.Dropout(0.5)(x)
                x = layers.Dense(512, activation='relu')(x)
                x = layers.Dropout(0.3)(x)
                outputs = layers.Dense(self.num_classes, activation='softmax')(x)

                self.model = keras.Model(inputs, outputs)
            else:
                self.model = base_model
        else:
            # Custom DenseNet kullan
            custom_densenet = CustomDenseNet(self.input_shape, self.num_classes)
            self.model = custom_densenet.build_model()

        return self.model

    def compile_for_recall(self,
                          loss_type: str = 'focal',
                          learning_rate: float = 0.001,
                          class_weights: Optional[Dict] = None):
        """
        Recall optimizasyonu için model compile et
        """
        # Loss function seç
        if loss_type == 'focal':
            loss_fn = RecallOptimizer.focal_loss(alpha=0.25, gamma=2.0)
        elif loss_type == 'weighted_bce':
            pos_weight = 2.0 if class_weights is None else class_weights[1]
            loss_fn = RecallOptimizer.weighted_binary_crossentropy(pos_weight)
        else:
            loss_fn = 'categorical_crossentropy'

        # Optimizer
        optimizer = optimizers.Adam(learning_rate=learning_rate)

        # Metrics
        metrics = [
            'accuracy',
            RecallOptimizer.recall_metric(threshold=0.5),
            keras.metrics.Precision(),
            keras.metrics.Recall()
        ]

        self.model.compile(
            optimizer=optimizer,
            loss=loss_fn,
            metrics=metrics
        )

        return self.model

    def create_callbacks(self,
                        monitor_metric: str = 'val_recall',
                        patience: int = 10) -> List[callbacks.Callback]:
        """
        Training callback'leri oluştur
        """
        callback_list = [
            callbacks.EarlyStopping(
                monitor=monitor_metric,
                patience=patience,
                restore_best_weights=True,
                mode='max'  # Recall maksimize etmek için
            ),
            callbacks.ReduceLROnPlateau(
                monitor=monitor_metric,
                factor=0.5,
                patience=5,
                min_lr=1e-7,
                mode='max'
            ),
            callbacks.ModelCheckpoint(
                'best_densenet_recall.h5',
                monitor=monitor_metric,
                save_best_only=True,
                mode='max'
            )
        ]

        return callback_list

    def train(self,
              train_data,
              validation_data,
              epochs: int = 100,
              batch_size: int = 32,
              class_weights: Optional[Dict] = None) -> keras.callbacks.History:
        """
        Model eğitimi
        """
        callbacks_list = self.create_callbacks()

        self.history = self.model.fit(
            train_data,
            validation_data=validation_data,
            epochs=epochs,
            batch_size=batch_size,
            callbacks=callbacks_list,
            class_weight=class_weights,
            verbose=1
        )

        return self.history

    def evaluate_recall(self, test_data, threshold: float = 0.5):
        """
        Test seti üzerinde recall ve diğer metrikleri değerlendir
        """
        # Predictions al
        predictions = self.model.predict(test_data)
        y_pred_binary = (predictions > threshold).astype(int)

        # True labels al (generator'dan)
        y_true = []
        for i in range(len(test_data)):
            batch_x, batch_y = test_data[i]
            y_true.extend(batch_y)
        y_true = np.array(y_true)

        # Classification report
        report = classification_report(y_true.argmax(axis=1),
                                     y_pred_binary.argmax(axis=1),
                                     target_names=['Negative', 'Positive'])
        print("Classification Report:")
        print(report)

        # Confusion Matrix
        cm = confusion_matrix(y_true.argmax(axis=1),
                            y_pred_binary.argmax(axis=1))

        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.title('Confusion Matrix')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.show()

        return report, cm

    def plot_precision_recall_curve(self, test_data):
        """
        Precision-Recall curve çiz
        """
        # Predictions ve true labels al
        predictions = self.model.predict(test_data)

        y_true = []
        for i in range(len(test_data)):
            batch_x, batch_y = test_data[i]
            y_true.extend(batch_y)
        y_true = np.array(y_true)

        # Pozitif sınıf için curve çiz
        precision, recall, thresholds = precision_recall_curve(
            y_true[:, 1], predictions[:, 1]
        )

        plt.figure(figsize=(10, 6))
        plt.plot(recall, precision, linewidth=2)
        plt.xlabel('Recall')
        plt.ylabel('Precision')
        plt.title('Precision-Recall Curve')
        plt.grid(True)
        plt.show()

        return precision, recall, thresholds

    def plot_training_history(self):
        """
        Training history'yi çiz
        """
        if self.history is None:
            print("Model henüz eğitilmedi!")
            return

        fig, axes = plt.subplots(2, 2, figsize=(15, 10))

        # Loss
        axes[0, 0].plot(self.history.history['loss'], label='Train Loss')
        axes[0, 0].plot(self.history.history['val_loss'], label='Val Loss')
        axes[0, 0].set_title('Model Loss')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Loss')
        axes[0, 0].legend()

        # Accuracy
        axes[0, 1].plot(self.history.history['accuracy'], label='Train Acc')
        axes[0, 1].plot(self.history.history['val_accuracy'], label='Val Acc')
        axes[0, 1].set_title('Model Accuracy')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Accuracy')
        axes[0, 1].legend()

        # Recall
        axes[1, 0].plot(self.history.history['recall'], label='Train Recall')
        axes[1, 0].plot(self.history.history['val_recall'], label='Val Recall')
        axes[1, 0].set_title('Model Recall')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Recall')
        axes[1, 0].legend()

        # Precision
        axes[1, 1].plot(self.history.history['precision'], label='Train Precision')
        axes[1, 1].plot(self.history.history['val_precision'], label='Val Precision')
        axes[1, 1].set_title('Model Precision')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Precision')
        axes[1, 1].legend()

        plt.tight_layout()
        plt.show()

# Kullanım örneği
def example_usage():
    """
    DenseNet ile recall optimizasyonu örnek kullanımı
    """

    # 1. Model oluştur
    trainer = DenseNetTrainer(
        input_shape=(224, 224, 3),
        num_classes=2,
        use_pretrained=True
    )

    # 2. Model compile et
    model = trainer.create_model(model_type='densenet121')

    # Class weights (imbalanced dataset için)
    class_weights = {0: 1.0, 1: 3.0}  # Pozitif sınıfa 3x ağırlık

    trainer.compile_for_recall(
        loss_type='focal',
        learning_rate=0.001,
        class_weights=class_weights
    )

    print("Model özeti:")
    print(model.summary())

    # 3. Data generators (örnek - gerçek data ile değiştirin)
    # train_generator = your_train_data_generator
    # val_generator = your_validation_data_generator
    # test_generator = your_test_data_generator

    # 4. Model eğitimi
    # history = trainer.train(
    #     train_data=train_generator,
    #     validation_data=val_generator,
    #     epochs=50,
    #     batch_size=32,
    #     class_weights=class_weights
    # )

    # 5. Training history görselleştir
    # trainer.plot_training_history()

    # 6. Test seti değerlendirmesi
    # report, cm = trainer.evaluate_recall(test_generator)

    # 7. Precision-Recall curve
    # precision, recall, thresholds = trainer.plot_precision_recall_curve(test_generator)

    print("DenseNet recall optimizasyonu kurulumu tamamlandı!")
    print("Gerçek veri ile eğitim için yorum satırlarını açın.")

if __name__ == "__main__":
    example_usage()

29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Model özeti:


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       524,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │         1,026 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,563,330 (28.85 MB)

 Trainable params: 7,479,682 (28.53 MB)

 Non-trainable params: 83,648 (326.75 KB)

None
DenseNet recall optimizasyonu kurulumu tamamlandı!
Gerçek veri ile eğitim için yorum satırlarını açın.
